# Payment Assistant -- Colab T4 Benchmark

Runs the **full-quality** tier (hybrid retrieval + cross-encoder re-ranking + `qwen2.5:7b-instruct`) end to end on a free Colab GPU runtime, and produces one comparison row for the "Benchmarks" table in the repo's `README.md` (full methodology in `DECISIONS.md` D26).

**Before running:** `Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`, then `Runtime -> Run all`.

This notebook is self-contained: it clones the repo, installs Ollama and the Python dependencies, pulls the model, generates and ingests the synthetic corpus, and runs `scripts/benchmark.py` -- the exact same script that produced the local (RTX 4060 Laptop, 8 GB) rows in `README.md`, so the two are directly comparable.

## 1. Confirm a GPU runtime is selected

In [ ]:
import subprocess

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch

assert torch.cuda.is_available(), (
    'No CUDA GPU visible. In Colab: Runtime -> Change runtime type -> '
    'Hardware accelerator -> T4 GPU, then Runtime -> Restart and run all.'
)
print('GPU OK:', torch.cuda.get_device_name(0))

## 2. Clone the repository

In [ ]:
import os

REPO_URL = 'https://github.com/ism00efe/Staj-projesi.git'
REPO_DIR = '/content/Staj-projesi'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'{REPO_DIR} already present, skipping clone.')

%cd {REPO_DIR}

## 3. Install Ollama and the Python dependencies

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# -e ".[dev]" installs the app itself plus pytest/httpx/numpy -- benchmark.py needs numpy,
# which ships as a core dependency, but installing the dev extra matches every other
# script/eval tool in this repo and keeps the setup command copy-pasteable everywhere.
!pip install -e ".[dev]" -q

## 4. Start Ollama and pull the model

In [ ]:
import subprocess
import time
import urllib.request

ollama_process = subprocess.Popen(
    ['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
        print('Ollama server is up.')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama server did not respond after 60s.')

!ollama pull qwen2.5:7b-instruct

## 5. Generate the synthetic corpus and ingest it

In [ ]:
!python scripts/generate_data.py
!python scripts/ingest.py

## 6. Run the benchmark (full-quality tier, GPU re-ranker enabled)

Reuses `scripts/benchmark.py` -- the same script, same tier definitions, same `eval/dataset.jsonl` -- so this records the exact same metrics as the local run in `README.md`: recall@5, MRR, citation precision, groundedness, p50/p95 latency, and peak GPU memory.

In [ ]:
!python scripts/benchmark.py --tier full-quality --hardware-label "Colab T4, 16 GB" --output benchmark_results_colab.md

## 7. Show the comparison row

In [ ]:
with open('benchmark_results_colab.md', encoding='utf-8') as f:
    print(f.read())

## Next step

Copy the row above (**Hardware = "Colab T4, 16 GB"**) into the "Benchmarks" section of the repo's `README.md`, next to the locally measured rows. Full methodology and rationale: `DECISIONS.md` D26.